In [49]:
import pandas as pd
import numpy as np
from dateutil import parser
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import requests
import re

In [50]:
url = "https://jiji.ng/api_web/v1/listing?slug=houses-apartments-for-sale&page=1&webp=true"
response = requests.get(url)

data = response.json()

 Nigeria Jiji Housing Marketplace Data Analysis
Title:
Market Analysis of Housing Sales on Jiji.ng Using Python
Overview
You will analyze real-world housing listings from the Jiji Nigeria Marketplace.
Your task is to extract listing data using Python from Jiji’s public API, clean and preprocess the data, perform exploratory data analysis (EDA), visualize the trends, and generate business insights and recommendations.
This assignment will help you practice:
Working with APIs

JSON data extraction

Data cleaning and transformation

Exploratory data analysis

Visualization

Business insight generation

Task 1: Data Extraction
Objective:
Extract housing data from the Jiji API and convert it into a clean DataFrame.
API Endpoint to Use:
https://jiji.ng/api_web/v1/listing?slug=houses-apartments-for-sale&page=1&webp=true

Requirements:
Write a Python script to send a GET request to the endpoint.

Use the requests library.

Extract the following key fields from each advert object:

title

region, region_name, region_parent_name

price_title (formatted price)

attrs fields

Property size

Bedrooms

Bathrooms

Furnishing status

is_boost (enterprise/diamond)

Convert the extracted data into a Pandas DataFrame.
Save the dataset locally as jiji_housing_raw.csv.



In [51]:
records = []

for item in data["adverts_list"]["adverts"]:

    attrs = item.get("attrs", [])

    bedrooms = None
    bathrooms = None
    property_size = None
    furnishing = None

    for attr in attrs:

        name = attr.get("name","")

        if name == "Bedrooms":
            bedrooms = attr.get("value")

        elif name == "Bathrooms":
            bathrooms = attr.get("value")

        elif name == "Property size":
            property_size = attr.get("value")

        elif name == "Furnishing":
            furnishing = attr.get("value")

    records.append({

        "title":item.get("title"),

        "region":item.get("region"),

        "region_name":item.get("region_name"),

        "region_parent_name":item.get("region_parent_name"),

        "price_title":item.get("price_title"),

        "property_size":property_size,

        "bedrooms":bedrooms,

        "bathrooms":bathrooms,

        "furnishing":furnishing,

        "is_boost":item.get("is_boost")

    })
df = pd.DataFrame(records)
df.to_csv("data/jiji_housing_raw.csv", index=False)
df.head(20)

,title,region,region_name,region_parent_name,price_title,property_size,bedrooms,bathrooms,furnishing,is_boost
0,2bdrm Block of Flats in Ojoo Abeokuta North fo...,"Ogun State, Abeokuta North",Abeokuta North,Ogun State,"₦ 7,500,000",1600,2,2,Unfurnished,False
1,"Furnished 5bdrm Duplex in Hampton Bay Estate, ...","Lekki, Ikate",Ikate,Lekki,"₦ 850,000,000",500,5,6,Furnished,vip_gold
2,4bdrm Duplex in Lekki Phase 2 for sale,"Lekki, Lekki Phase 2",Lekki Phase 2,Lekki,"₦ 220,000,000",450,4,4,Semi-Furnished,premium
3,Furnished 4bdrm Duplex in Lekki for sale,"Lagos State, Lekki",Lekki,Lagos State,"₦ 600,000,000",350,4,4,Furnished,enterprise
4,Furnished 3bdrm Bungalow in Ijebu Ode for sale,"Ogun State, Ijebu Ode",Ijebu Ode,Ogun State,"₦ 85,000,000",600,3,3,Furnished,False
5,2bdrm Apartment in Ikate for sale,"Lekki, Ikate",Ikate,Lekki,"₦ 120,000,000",500,2,2,Unfurnished,premium
6,4bdrm House in Victoria Island Extension for sale,"Victoria Island, Victoria Island Extension",Victoria Island Extension,Victoria Island,"₦ 800,000,000",1200,4,4,Semi-Furnished,diamond
7,"8bdrm Duplex in Ikolaba Estate, Agodi for sale","Ibadan, Agodi",Agodi,Ibadan,"₦ 250,000,000",922,8,6,Unfurnished,False
8,1bdrm Block of Flats in Ikorodu for sale,"Lagos State, Ikorodu",Ikorodu,Lagos State,"₦ 230,000,000",854,1,1,Unfurnished,False
9,Furnished 2bdrm Duplex in Fo1 Kubwa for sale,"Abuja (FCT), Kubwa",Kubwa,Abuja (FCT),"₦ 100,000,000",1000,2,3,Furnished,vip_gold


2. Data Cleaning
Objective:
Prepare the dataset for reliable analysis.
Questions to Guide Cleaning:
Missing Values

Which columns have missing values?

How will you handle missing property size, bedrooms, bathrooms, or furnishing?

Price Cleaning

Convert the price from strings like “₦ 85,500,000” to float (85500000.00)

Are there outlier prices that should be removed?

Categorical Cleaning

Standardize region, region_name, region_parent_name.

Convert furnishing type to consistent categories
 (e.g., Furnished / Semi-furnished / Unfurnished / Unknown)

Attributes Extraction

Extract values for Bedrooms, Bathrooms, Property size from the attrs JSON list.

Remove duplicates

Look for duplicate titles or identical rows.

Save cleaned dataset as:
 jiji_housing_cleaned.csv

In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   title               20 non-null     str   
 1   region              20 non-null     str   
 2   region_name         20 non-null     str   
 3   region_parent_name  19 non-null     str   
 4   price_title         20 non-null     str   
 5   property_size       20 non-null     int64 
 6   bedrooms            20 non-null     int64 
 7   bathrooms           20 non-null     int64 
 8   furnishing          20 non-null     str   
 9   is_boost            20 non-null     object
dtypes: int64(3), object(1), str(6)
memory usage: 3.7+ KB


In [53]:
df.isnull().sum()

title                 0
region                0
region_name           0
region_parent_name    1
price_title           0
property_size         0
bedrooms              0
bathrooms             0
furnishing            0
is_boost              0
dtype: int64

In [54]:
df.describe()

,property_size,bedrooms,bathrooms
count,20.000000,20.000000,20.000000
mean,718.550000,3.500000,3.550000
std,677.590272,1.849609,1.538112
min,60.000000,1.000000,1.000000
25%,318.750000,2.000000,2.750000
50%,500.000000,3.000000,3.500000
75%,941.500000,4.250000,4.250000
max,3000.000000,8.000000,6.000000


In [55]:
#checking for missing values

df["bedrooms"] = df["bedrooms"].fillna(0)

df["bathrooms"] = df["bathrooms"].fillna(0)

df["furnishing"] = df["furnishing"].fillna("Unknown")

df["property_size"] = df["property_size"].fillna("Unknown")
df

,title,region,region_name,region_parent_name,price_title,property_size,bedrooms,bathrooms,furnishing,is_boost
0,2bdrm Block of Flats in Ojoo Abeokuta North fo...,"Ogun State, Abeokuta North",Abeokuta North,Ogun State,"₦ 7,500,000",1600,2,2,Unfurnished,False
1,"Furnished 5bdrm Duplex in Hampton Bay Estate, ...","Lekki, Ikate",Ikate,Lekki,"₦ 850,000,000",500,5,6,Furnished,vip_gold
2,4bdrm Duplex in Lekki Phase 2 for sale,"Lekki, Lekki Phase 2",Lekki Phase 2,Lekki,"₦ 220,000,000",450,4,4,Semi-Furnished,premium
3,Furnished 4bdrm Duplex in Lekki for sale,"Lagos State, Lekki",Lekki,Lagos State,"₦ 600,000,000",350,4,4,Furnished,enterprise
4,Furnished 3bdrm Bungalow in Ijebu Ode for sale,"Ogun State, Ijebu Ode",Ijebu Ode,Ogun State,"₦ 85,000,000",600,3,3,Furnished,False
5,2bdrm Apartment in Ikate for sale,"Lekki, Ikate",Ikate,Lekki,"₦ 120,000,000",500,2,2,Unfurnished,premium
6,4bdrm House in Victoria Island Extension for sale,"Victoria Island, Victoria Island Extension",Victoria Island Extension,Victoria Island,"₦ 800,000,000",1200,4,4,Semi-Furnished,diamond
7,"8bdrm Duplex in Ikolaba Estate, Agodi for sale","Ibadan, Agodi",Agodi,Ibadan,"₦ 250,000,000",922,8,6,Unfurnished,False
8,1bdrm Block of Flats in Ikorodu for sale,"Lagos State, Ikorodu",Ikorodu,Lagos State,"₦ 230,000,000",854,1,1,Unfurnished,False
9,Furnished 2bdrm Duplex in Fo1 Kubwa for sale,"Abuja (FCT), Kubwa",Kubwa,Abuja (FCT),"₦ 100,000,000",1000,2,3,Furnished,vip_gold


In [56]:
df["price_title"] = (
    df["price_title"]
    .astype(str)
    .str.replace("₦", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price_title"] = pd.to_numeric(df["price_title"], errors="coerce")

df

,title,region,region_name,region_parent_name,price_title,property_size,bedrooms,bathrooms,furnishing,is_boost
0,2bdrm Block of Flats in Ojoo Abeokuta North fo...,"Ogun State, Abeokuta North",Abeokuta North,Ogun State,7500000,1600,2,2,Unfurnished,False
1,"Furnished 5bdrm Duplex in Hampton Bay Estate, ...","Lekki, Ikate",Ikate,Lekki,850000000,500,5,6,Furnished,vip_gold
2,4bdrm Duplex in Lekki Phase 2 for sale,"Lekki, Lekki Phase 2",Lekki Phase 2,Lekki,220000000,450,4,4,Semi-Furnished,premium
3,Furnished 4bdrm Duplex in Lekki for sale,"Lagos State, Lekki",Lekki,Lagos State,600000000,350,4,4,Furnished,enterprise
4,Furnished 3bdrm Bungalow in Ijebu Ode for sale,"Ogun State, Ijebu Ode",Ijebu Ode,Ogun State,85000000,600,3,3,Furnished,False
5,2bdrm Apartment in Ikate for sale,"Lekki, Ikate",Ikate,Lekki,120000000,500,2,2,Unfurnished,premium
6,4bdrm House in Victoria Island Extension for sale,"Victoria Island, Victoria Island Extension",Victoria Island Extension,Victoria Island,800000000,1200,4,4,Semi-Furnished,diamond
7,"8bdrm Duplex in Ikolaba Estate, Agodi for sale","Ibadan, Agodi",Agodi,Ibadan,250000000,922,8,6,Unfurnished,False
8,1bdrm Block of Flats in Ikorodu for sale,"Lagos State, Ikorodu",Ikorodu,Lagos State,230000000,854,1,1,Unfurnished,False
9,Furnished 2bdrm Duplex in Fo1 Kubwa for sale,"Abuja (FCT), Kubwa",Kubwa,Abuja (FCT),100000000,1000,2,3,Furnished,vip_gold


In [57]:
#Are there any outliers or invalid prices (too low/high)?

Q1 = df['price_title'].quantile(0.25)
Q3 = df['price_title'].quantile(0.75)
IQR = Q3 - Q1
                    
IQR
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (df['price_title'] < lower_bound) | (df['price_title'] > upper_bound)

print(outliers.sum())
new_df = df[
    (df['price_title'] > lower_bound) & (df['price_title'] < upper_bound)
]
new_df

0


,title,region,region_name,region_parent_name,price_title,property_size,bedrooms,bathrooms,furnishing,is_boost
0,2bdrm Block of Flats in Ojoo Abeokuta North fo...,"Ogun State, Abeokuta North",Abeokuta North,Ogun State,7500000,1600,2,2,Unfurnished,False
1,"Furnished 5bdrm Duplex in Hampton Bay Estate, ...","Lekki, Ikate",Ikate,Lekki,850000000,500,5,6,Furnished,vip_gold
2,4bdrm Duplex in Lekki Phase 2 for sale,"Lekki, Lekki Phase 2",Lekki Phase 2,Lekki,220000000,450,4,4,Semi-Furnished,premium
3,Furnished 4bdrm Duplex in Lekki for sale,"Lagos State, Lekki",Lekki,Lagos State,600000000,350,4,4,Furnished,enterprise
4,Furnished 3bdrm Bungalow in Ijebu Ode for sale,"Ogun State, Ijebu Ode",Ijebu Ode,Ogun State,85000000,600,3,3,Furnished,False
5,2bdrm Apartment in Ikate for sale,"Lekki, Ikate",Ikate,Lekki,120000000,500,2,2,Unfurnished,premium
6,4bdrm House in Victoria Island Extension for sale,"Victoria Island, Victoria Island Extension",Victoria Island Extension,Victoria Island,800000000,1200,4,4,Semi-Furnished,diamond
7,"8bdrm Duplex in Ikolaba Estate, Agodi for sale","Ibadan, Agodi",Agodi,Ibadan,250000000,922,8,6,Unfurnished,False
8,1bdrm Block of Flats in Ikorodu for sale,"Lagos State, Ikorodu",Ikorodu,Lagos State,230000000,854,1,1,Unfurnished,False
9,Furnished 2bdrm Duplex in Fo1 Kubwa for sale,"Abuja (FCT), Kubwa",Kubwa,Abuja (FCT),100000000,1000,2,3,Furnished,vip_gold


3.  Cleaning


Standardize region, region_name, region_parent_name.


Convert furnishing type to consistent categories
 (e.g., Furnished / Semi-furnished / Unfurnished / Unknown)



In [58]:
new_df.rename(columns={
    "region": "Region",
    "region_name": "Region Name",
    "region_parent_name": "Region Parent Name"
}, inplace=True)
new_df

,title,Region,Region Name,Region Parent Name,price_title,property_size,bedrooms,bathrooms,furnishing,is_boost
0,2bdrm Block of Flats in Ojoo Abeokuta North fo...,"Ogun State, Abeokuta North",Abeokuta North,Ogun State,7500000,1600,2,2,Unfurnished,False
1,"Furnished 5bdrm Duplex in Hampton Bay Estate, ...","Lekki, Ikate",Ikate,Lekki,850000000,500,5,6,Furnished,vip_gold
2,4bdrm Duplex in Lekki Phase 2 for sale,"Lekki, Lekki Phase 2",Lekki Phase 2,Lekki,220000000,450,4,4,Semi-Furnished,premium
3,Furnished 4bdrm Duplex in Lekki for sale,"Lagos State, Lekki",Lekki,Lagos State,600000000,350,4,4,Furnished,enterprise
4,Furnished 3bdrm Bungalow in Ijebu Ode for sale,"Ogun State, Ijebu Ode",Ijebu Ode,Ogun State,85000000,600,3,3,Furnished,False
5,2bdrm Apartment in Ikate for sale,"Lekki, Ikate",Ikate,Lekki,120000000,500,2,2,Unfurnished,premium
6,4bdrm House in Victoria Island Extension for sale,"Victoria Island, Victoria Island Extension",Victoria Island Extension,Victoria Island,800000000,1200,4,4,Semi-Furnished,diamond
7,"8bdrm Duplex in Ikolaba Estate, Agodi for sale","Ibadan, Agodi",Agodi,Ibadan,250000000,922,8,6,Unfurnished,False
8,1bdrm Block of Flats in Ikorodu for sale,"Lagos State, Ikorodu",Ikorodu,Lagos State,230000000,854,1,1,Unfurnished,False
9,Furnished 2bdrm Duplex in Fo1 Kubwa for sale,"Abuja (FCT), Kubwa",Kubwa,Abuja (FCT),100000000,1000,2,3,Furnished,vip_gold


In [59]:
new_df["furnishing"] = new_df["furnishing"].replace({

"Fully Furnished":"Furnished",

"Semi Furnished":"Semi-furnished",

"Unfurnished":"Unfurnished"

})
new_df


,title,Region,Region Name,Region Parent Name,price_title,property_size,bedrooms,bathrooms,furnishing,is_boost
0,2bdrm Block of Flats in Ojoo Abeokuta North fo...,"Ogun State, Abeokuta North",Abeokuta North,Ogun State,7500000,1600,2,2,Unfurnished,False
1,"Furnished 5bdrm Duplex in Hampton Bay Estate, ...","Lekki, Ikate",Ikate,Lekki,850000000,500,5,6,Furnished,vip_gold
2,4bdrm Duplex in Lekki Phase 2 for sale,"Lekki, Lekki Phase 2",Lekki Phase 2,Lekki,220000000,450,4,4,Semi-Furnished,premium
3,Furnished 4bdrm Duplex in Lekki for sale,"Lagos State, Lekki",Lekki,Lagos State,600000000,350,4,4,Furnished,enterprise
4,Furnished 3bdrm Bungalow in Ijebu Ode for sale,"Ogun State, Ijebu Ode",Ijebu Ode,Ogun State,85000000,600,3,3,Furnished,False
5,2bdrm Apartment in Ikate for sale,"Lekki, Ikate",Ikate,Lekki,120000000,500,2,2,Unfurnished,premium
6,4bdrm House in Victoria Island Extension for sale,"Victoria Island, Victoria Island Extension",Victoria Island Extension,Victoria Island,800000000,1200,4,4,Semi-Furnished,diamond
7,"8bdrm Duplex in Ikolaba Estate, Agodi for sale","Ibadan, Agodi",Agodi,Ibadan,250000000,922,8,6,Unfurnished,False
8,1bdrm Block of Flats in Ikorodu for sale,"Lagos State, Ikorodu",Ikorodu,Lagos State,230000000,854,1,1,Unfurnished,False
9,Furnished 2bdrm Duplex in Fo1 Kubwa for sale,"Abuja (FCT), Kubwa",Kubwa,Abuja (FCT),100000000,1000,2,3,Furnished,vip_gold


4. Attributes Extraction


Extract values for Bedrooms, Bathrooms, Property size from the attrs JSON list.



In [60]:
def extract_size(x):

    if pd.isna(x):
        return None

    nums = re.findall(r"\d+", str(x))

    if nums:
        return float(nums[0])

    return None

new_df["property_size"] = new_df["property_size"].apply(extract_size)
new_df

,title,Region,Region Name,Region Parent Name,price_title,property_size,bedrooms,bathrooms,furnishing,is_boost
0,2bdrm Block of Flats in Ojoo Abeokuta North fo...,"Ogun State, Abeokuta North",Abeokuta North,Ogun State,7500000,1600.0,2,2,Unfurnished,False
1,"Furnished 5bdrm Duplex in Hampton Bay Estate, ...","Lekki, Ikate",Ikate,Lekki,850000000,500.0,5,6,Furnished,vip_gold
2,4bdrm Duplex in Lekki Phase 2 for sale,"Lekki, Lekki Phase 2",Lekki Phase 2,Lekki,220000000,450.0,4,4,Semi-Furnished,premium
3,Furnished 4bdrm Duplex in Lekki for sale,"Lagos State, Lekki",Lekki,Lagos State,600000000,350.0,4,4,Furnished,enterprise
4,Furnished 3bdrm Bungalow in Ijebu Ode for sale,"Ogun State, Ijebu Ode",Ijebu Ode,Ogun State,85000000,600.0,3,3,Furnished,False
5,2bdrm Apartment in Ikate for sale,"Lekki, Ikate",Ikate,Lekki,120000000,500.0,2,2,Unfurnished,premium
6,4bdrm House in Victoria Island Extension for sale,"Victoria Island, Victoria Island Extension",Victoria Island Extension,Victoria Island,800000000,1200.0,4,4,Semi-Furnished,diamond
7,"8bdrm Duplex in Ikolaba Estate, Agodi for sale","Ibadan, Agodi",Agodi,Ibadan,250000000,922.0,8,6,Unfurnished,False
8,1bdrm Block of Flats in Ikorodu for sale,"Lagos State, Ikorodu",Ikorodu,Lagos State,230000000,854.0,1,1,Unfurnished,False
9,Furnished 2bdrm Duplex in Fo1 Kubwa for sale,"Abuja (FCT), Kubwa",Kubwa,Abuja (FCT),100000000,1000.0,2,3,Furnished,vip_gold


5. Remove duplicates


Look for duplicate titles or identical rows.




In [61]:
new_df.drop_duplicates(inplace=True)
new_df

,title,Region,Region Name,Region Parent Name,price_title,property_size,bedrooms,bathrooms,furnishing,is_boost
0,2bdrm Block of Flats in Ojoo Abeokuta North fo...,"Ogun State, Abeokuta North",Abeokuta North,Ogun State,7500000,1600.0,2,2,Unfurnished,False
1,"Furnished 5bdrm Duplex in Hampton Bay Estate, ...","Lekki, Ikate",Ikate,Lekki,850000000,500.0,5,6,Furnished,vip_gold
2,4bdrm Duplex in Lekki Phase 2 for sale,"Lekki, Lekki Phase 2",Lekki Phase 2,Lekki,220000000,450.0,4,4,Semi-Furnished,premium
3,Furnished 4bdrm Duplex in Lekki for sale,"Lagos State, Lekki",Lekki,Lagos State,600000000,350.0,4,4,Furnished,enterprise
4,Furnished 3bdrm Bungalow in Ijebu Ode for sale,"Ogun State, Ijebu Ode",Ijebu Ode,Ogun State,85000000,600.0,3,3,Furnished,False
5,2bdrm Apartment in Ikate for sale,"Lekki, Ikate",Ikate,Lekki,120000000,500.0,2,2,Unfurnished,premium
6,4bdrm House in Victoria Island Extension for sale,"Victoria Island, Victoria Island Extension",Victoria Island Extension,Victoria Island,800000000,1200.0,4,4,Semi-Furnished,diamond
7,"8bdrm Duplex in Ikolaba Estate, Agodi for sale","Ibadan, Agodi",Agodi,Ibadan,250000000,922.0,8,6,Unfurnished,False
8,1bdrm Block of Flats in Ikorodu for sale,"Lagos State, Ikorodu",Ikorodu,Lagos State,230000000,854.0,1,1,Unfurnished,False
9,Furnished 2bdrm Duplex in Fo1 Kubwa for sale,"Abuja (FCT), Kubwa",Kubwa,Abuja (FCT),100000000,1000.0,2,3,Furnished,vip_gold


6. Save cleaned dataset as:
 jiji_housing_cleaned.csv




In [62]:
new_df.to_csv("jiji_housing_cleaned.csv", index=False)

Task 3: Exploratory Data Analysis (EDA)
Objective:
Understand the structure, patterns, and distribution of the Nigerian housing marketplace on Jiji.
Your EDA Should Answer These Questions:
Price Analysis


What is the average house price in Nigeria?


Which state has the highest and lowest mean property price?




In [63]:
new_df['price_title'].mean()

np.float64(300825000.0)

In [64]:
state_price = (
    new_df.groupby("Region Name")["price_title"]
    .mean()
    .sort_values(ascending=False)
)
print("Highest Mean Price State:")
print(state_price.head())

print("\nLowest Mean Price State:")
print(state_price.tail())

Highest Mean Price State:
Region Name
Victoria Island Extension    800000000.0
Lekki                        600000000.0
Ikate                        485000000.0
Ikeja GRA                    400000000.0
Lekki Phase 2                385000000.0
Name: price_title, dtype: float64

Lowest Mean Price State:
Region Name
Ijebu Ode         85000000.0
Erunwe            42000000.0
Owerri            30000000.0
Osun State        12000000.0
Abeokuta North     7500000.0
Name: price_title, dtype: float64


Property Size & Features


What is the distribution of property sizes?


Do houses with more bedrooms/bathrooms cost significantly more?




In [65]:
new_df['property_size'].value_counts()

property_size
500.0     2
450.0     2
600.0     2
1200.0    2
200.0     2
60.0      2
1600.0    1
350.0     1
922.0     1
854.0     1
1000.0    1
300.0     1
3000.0    1
325.0     1
Name: count, dtype: int64

In [66]:
new_df.groupby("bedrooms")["price_title"].mean()

bedrooms
1    130000000.0
2    205500000.0
3    134750000.0
4    492500000.0
5    560000000.0
7    290000000.0
8    250000000.0
Name: price_title, dtype: float64

Regional Trends


Which regions have the highest number of listings?


Which regions dominate premium property sales?



In [67]:
region_counts = new_df['Region Name'].value_counts()
print(region_counts.head(20))

Region Name
Ikate                        2
Lekki Phase 2                2
Kubwa                        2
Abeokuta North               1
Lekki                        1
Ijebu Ode                    1
Victoria Island Extension    1
Agodi                        1
Ikorodu                      1
Ikeja GRA                    1
Osun State                   1
Owerri                       1
Maitama                      1
Ikota                        1
Idu Industrial               1
Life Camp                    1
Erunwe                       1
Name: count, dtype: int64


In [68]:
new_df.groupby("Region Name")["price_title"].mean().sort_values(ascending=False)

Region Name
Victoria Island Extension    800000000.0
Lekki                        600000000.0
Ikate                        485000000.0
Ikeja GRA                    400000000.0
Lekki Phase 2                385000000.0
Ikota                        350000000.0
Kubwa                        350000000.0
Idu Industrial               290000000.0
Life Camp                    280000000.0
Agodi                        250000000.0
Ikorodu                      230000000.0
Maitama                      200000000.0
Ijebu Ode                     85000000.0
Erunwe                        42000000.0
Owerri                        30000000.0
Osun State                    12000000.0
Abeokuta North                 7500000.0
Name: price_title, dtype: float64

Furnishing Analysis


Are furnished apartments more expensive on average?



In [69]:
new_df.groupby("furnishing")["price_title"].mean()

furnishing
Furnished         4.308333e+08
Semi-Furnished    3.700000e+08
Unfurnished       1.757222e+08
Name: price_title, dtype: float64

Listing Type


Do boosted/enterprise listings have higher prices?



In [70]:
new_df.groupby("is_boost")["price_title"].mean()

is_boost
False         1.718333e+08
diamond       8.000000e+08
enterprise    4.000000e+08
premium       3.040000e+08
vip_gold      4.500000e+08
Name: price_title, dtype: float64

Task 5: Summary of Findings 
Rewrite your findings by answering the following:
Which states have the highest-priced listings?


Which states are the most affordable?


Do furnished homes consistently cost more than unfurnished ones?


How strongly do bedrooms, bathrooms, or property size influence price?


Are boosted (Enterprise) listings typically more expensive?


What property types attract premium pricing?


What patterns exist in Lagos vs Abuja vs other regions?



The highest-priced property listings are concentrated in premium markets such as Lagos and the Federal Capital Territory (Abuja).

States with fewer luxury developments generally have the lowest average property prices.

Furnished homes tend to have higher average prices than unfurnished homes, although the difference varies across regions.

Property price generally increases with the number of bedrooms and bathrooms, showing a positive relationship between size and value.

Larger properties typically command higher prices, but location remains a major pricing factor.

Enterprise or boosted listings often have higher average prices because they usually represent premium properties or professional agencies.

Premium pricing is commonly associated with luxury apartments, detached duplexes, and high-end estates.

Lagos and Abuja dominate the luxury housing market, while many other states focus on more affordable housing options.


Task 6: Business Insights & Recommendations
Provide a 5–7 line business-focused insight summary, answering:
Which housing type or feature performs best overall on Jiji?


Which state contributes the most revenue potential?


Which states underperform?


Which period (if using timestamps) records the highest listing activity?


What strategies can improve sales in low-performing regions?


What can sellers do to increase listing visibility and pricing power?




Luxury properties with more bedrooms, larger sizes, and furnished interiors generate the strongest pricing performance on Jiji.

Lagos contributes the greatest revenue potential due to its high listing volume and premium property prices, followed by Abuja.

Lower-performing states may require improved marketing and competitive pricing strategies to attract buyers.

If timestamp data is available, listing activity should be analyzed to identify seasonal peaks and optimize advertising campaigns.

Sellers can improve visibility by using boosted listings, providing detailed property information, and uploading high-quality images.


Agencies should prioritize premium urban markets while tailoring affordable housing promotions for emerging regions to maximize overall sales potential.

In [48]:
new_df.to_csv("data/nigerian-housing-cleaned-dataset.csv", index=False)